<a href="https://colab.research.google.com/github/timraiswell/ai-engineer/blob/main/01-model-apis/04-context-and-caching.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Context Windows & Cost Control

**Goal:** Budget the context window, estimate cost, and understand caching + batch pricing (concepts you'll meet on any provider).

Part of [ai-engineer-notebooks](https://github.com/calmrocks/ai-engineer-notebooks), the hands-on companion to the [FDE / AI Engineer transition plan](https://www.calm.rocks/resources/career-development/transition-fde-ai-engineer/).


## Setup

Each notebook is self-contained, so the next two cells stand it up from scratch:

1. **Install dependencies.** The `aien` package (this repo) carries the shared setup helper and pulls in the `groq` client; `tiktoken` is used by this notebook.
2. **Load your API key.** Get a free key at [console.groq.com](https://console.groq.com/) (no credit card). In Colab, add it via the **key icon** in the left sidebar → **Add new secret**, name it exactly `GROQ_API_KEY`, paste the value, and toggle **Notebook access** on. Running locally instead? Set `GROQ_API_KEY` as an environment variable.

(Full walkthrough and model-picking guidance live in [00-setup/00-environment.ipynb](https://colab.research.google.com/github/calmrocks/ai-engineer-notebooks/blob/main/00-setup/00-environment.ipynb).)

In [1]:
%pip install -q "git+https://github.com/calmrocks/ai-engineer-notebooks.git" tiktoken

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 5.5 MB/s eta 0:00:00


In [2]:
from aien import setup

# Loads GROQ_API_KEY (Colab Secrets or local env var) and returns a ready
# Groq client. Pass model=... to override the default; if a call later 404s,
# list available models — see 00-setup/00-environment.ipynb.
client, MODEL = setup()

Groq client ready. MODEL = openai/gpt-oss-120b


## The mental model: a per-request token budget

Every request spends from one budget: the context window (128k tokens on `gpt-oss-120b`, though you'll rarely want to fill it; cost and latency scale with what you send). Think of each request as a ledger with four lines:

| Line | Typical share | Notes |
|---|---|---|
| System prompt | fixed, small-ish | instructions, persona, tool definitions |
| Retrieved context | the big variable | RAG chunks, documents, code |
| Conversation history | grows every turn | the thing that eventually eats everything |
| Output headroom | reserve it! | `max_tokens` counts against the window |

The failure mode isn't hitting the hard limit. It's cost creep: an agent that resends 50k tokens of history to answer a 10-token question. Budgeting means measuring each line.

Groq's OpenAI-compatible API has no dedicated `count_tokens` endpoint. Two practical options:

1. **Exact, after the fact:** every response carries `usage.prompt_tokens`, the true count for what you sent. Free, but you only learn it *after* the call.
2. **Approximate, before the fact:** use `tiktoken` (OpenAI's tokenizer) as an *estimate*. The models here tokenize differently, so treat this as a ballpark (usually within ~10-20%) for pre-flight budget checks, then trust `usage` for the real number.

In [3]:
import tiktoken

# tiktoken is OpenAI's cl100k tokenizer, which won't exactly match the model you're
# calling — so this is an ESTIMATE, good enough for pre-flight budgeting. For the true
# count, read usage.prompt_tokens off a response.
_enc = tiktoken.get_encoding('cl100k_base')


def count(messages, system=None):
    """Approximate prompt tokens for a message list (+ optional system string)."""
    text = ''
    if system:
        text += system + '\n'
    for m in messages:
        content = m['content'] if isinstance(m['content'], str) else str(m['content'])
        text += content + '\n'
    return len(_enc.encode(text))


SYSTEM = 'You are a support assistant for AcmeDB. Answer only from the provided runbook. Cite section numbers.'

print('system + 1 short question (approx):',
      count([{'role': 'user', 'content': 'How do I rotate credentials?'}], system=SYSTEM))

# Same content, two phrasings — tokens are about bytes-of-text, not 'ideas'
terse = 'rotate creds acmedb how'
prose = ('Hello! I was wondering if you could kindly walk me through the process of '
         'rotating the credentials for our AcmeDB instance when you get a chance?')
print('terse phrasing: ', count([{'role': 'user', 'content': terse}]))
print('prose phrasing: ', count([{'role': 'user', 'content': prose}]))

# Now the exact count from the API, for comparison:
r = client.chat.completions.create(
    model=MODEL, max_tokens=5,
    messages=[{'role': 'system', 'content': SYSTEM},
              {'role': 'user', 'content': 'How do I rotate credentials?'}],
)
print('exact prompt_tokens from usage:', r.usage.prompt_tokens)

system + 1 short question (approx): 29
terse phrasing:  7
prose phrasing:  30
exact prompt_tokens from usage: 102


In [4]:
# A budget checker you can keep: does this request fit the plan?
def check_budget(system, context, history, max_output,
                 budget={'system': 2_000, 'context': 8_000, 'history': 4_000}):
    # All estimates (tiktoken), no API calls. Trust usage.prompt_tokens for the real number.
    lines = {
        'system':  count([], system=system),
        'context': count([{'role': 'user', 'content': context}]),
        'history': count(history) if history else 0,
    }
    for name, used in lines.items():
        cap = budget[name]
        flag = 'OK ' if used <= cap else 'OVER'
        print(f'{flag} {name:8s} {used:>7,} / {cap:,}')
    total = sum(lines.values()) + max_output
    print(f'    total incl. {max_output:,} output headroom: {total:,} tokens')
    return lines


_ = check_budget(
    system=SYSTEM,
    context='Section 4.2: To rotate credentials, run acmedb rotate --all ...' * 40,
    history=[{'role': 'user', 'content': 'hi'}, {'role': 'assistant', 'content': 'Hello! How can I help?'}],
    max_output=1_000,
)


OK  system        23 / 2,000
OK  context      720 / 8,000
OK  history        9 / 4,000
    total incl. 1,000 output headroom: 1,752 tokens


## Prompt caching (a concept to know)

If every request re-sends the same big system prompt or document, you're paying full price to re-process bytes the model saw seconds ago. **Prompt caching** fixes that: the provider caches the processed prefix and charges a fraction to reuse it.

Groq does not currently expose a prompt-caching API, but you'll meet this everywhere else (Anthropic's `cache_control`, OpenAI's automatic prefix caching, Google's context caching), so it's worth understanding the economics:

- **It's a prefix match on exact bytes.** Any byte change *before* the cached point invalidates everything after it. Put volatile content (timestamps, user names) *after* the stable prefix, never inside it.
- **Typical economics:** a cache *write* costs ~1.25x the input rate; a cache *read* costs ~0.1x. Break-even after the second request, then a ~90% discount on the repeated prefix thereafter.
- **TTL is short** (often ~5 minutes, refreshed on each hit).

The cell below demonstrates the *problem* caching solves, on Groq: send the same 8k-token runbook prefix twice and watch `prompt_tokens`. You pay for every one of those tokens, every single call. On a caching provider the second call's prefix would be ~10x cheaper.


In [5]:
# Build a deterministic ~8k-token 'runbook'.
sections = []
for i in range(1, 121):
    sections.append(
        f'Section {i}: Procedure AC-{i:03d}. To perform maintenance task {i}, first '
        f'verify replica lag is under {i * 5} ms, then drain node group {i % 7}, apply '
        f'the configuration bundle, and re-enable traffic. Rollback: restore snapshot '
        f'tagged ac-{i:03d}-pre and replay the WAL from the checkpoint marker.'
    )
RUNBOOK = '\n'.join(sections)

print('runbook tokens (approx):',
      count([{'role': 'user', 'content': RUNBOOK}]))


runbook tokens (approx): 7440


In [6]:
def ask_runbook(question):
    return client.chat.completions.create(
        model=MODEL,
        max_tokens=150,
        messages=[
            {'role': 'system',
             'content': 'Answer from this runbook only, citing section numbers.\n\n' + RUNBOOK},
            {'role': 'user', 'content': question},
        ],
    )


# 2 API calls, same big prefix. Watch prompt_tokens: it's ~identical both times —
# you re-pay for the whole runbook on every call. THIS is what prompt caching removes.
for question in ['What is the rollback for procedure AC-042?',
                 'Which node group does task 10 drain?']:
    r = ask_runbook(question)
    u = r.usage
    print(f'Q: {question}')
    print(f'   prompt_tokens={u.prompt_tokens}  completion_tokens={u.completion_tokens}')
    print(f'   A: {r.choices[0].message.content[:110]}...')
    print()


Q: What is the rollback for procedure AC-042?
   prompt_tokens=7535  completion_tokens=146
   A: The rollback for procedure **AC‑042** is to **restore the snapshot tagged `ac-042-pre` and replay the WAL from...

Q: Which node group does task 10 drain?
   prompt_tokens=7534  completion_tokens=108
   A: Task 10 drains **node group 3**【Section 10】....



Run it and note that `prompt_tokens` is essentially the same on both calls: you paid for the full runbook twice. On a caching provider, the second call's big prefix would bill at ~10% of that. When you *do* have caching available, these silently zero your hit rate (the request still works, you just pay full price):

- A timestamp, request ID, or user name interpolated into the stable prefix makes the prefix differ every request. Keep volatile content *after* the cached section.
- Adding, removing, or reordering **tools**, which render ahead of the system prompt. Serialize them deterministically.
- Switching **models** mid-conversation, since caches are per-model.
- Non-deterministic serialization (`json.dumps` without `sort_keys`, iterating a set).

## The Batch API: half price for patience (another concept)

Real-time calls (everything above) bill at full rate. Most major providers also offer a **Batch API**: submit many requests, get results asynchronously (minutes to 24h), pay **~50%** of standard price. Groq doesn't offer one today, but the pattern maps to a specific set of FDE/AI-engineer jobs: **bulk evals** (re-score 500 transcripts), **backfills** (classify every historical ticket), nightly enrichment, regression sweeps. The tell is always the same: nobody is waiting on the output, and there are a lot of them. If a human is watching a spinner, batch is wrong; if it's a cron job, it's free money.

Since we can't run a batch on Groq, we'll do the *math* on when each lever pays off, the skill that actually matters in a design review.


### What these actually look like on a provider that has them

Groq can't run caching or batch, but you'll use both constantly on OpenAI/Anthropic, so here are the *real* API shapes as reference. **Don't run these** (they need those SDKs and paid keys); read them so the shapes aren't new to you the first time you need them.

**Anthropic prompt caching —** mark a cache breakpoint with `cache_control` on a content block; everything *before and including* it is cached (order is tools → system → messages). The `usage` fields tell you whether you got a hit.

```python
# pip install anthropic  — reference only, not run here
import anthropic
client = anthropic.Anthropic()

resp = client.messages.create(
    model="claude-sonnet-5",
    max_tokens=150,
    system=[
        {"type": "text", "text": "Answer from this runbook only, cite section numbers."},
        {"type": "text",
         "text": RUNBOOK,                              # the big, stable prefix
         "cache_control": {"type": "ephemeral"}},      # <-- cache everything up to here
    ],
    messages=[{"role": "user", "content": "What is the rollback for AC-042?"}],
)
u = resp.usage
# First call:  cache_creation_input_tokens = <runbook>,  cache_read_input_tokens = 0   (the ~1.25x write)
# Next call:   cache_creation_input_tokens = 0,          cache_read_input_tokens = <runbook>  (the ~0.1x read)
print(u.cache_creation_input_tokens, u.cache_read_input_tokens, u.input_tokens)
```

The discipline the earlier cells taught is exactly what makes this pay off: keep the volatile bytes (the user question) *after* the marked block, or every request is a cache miss. (OpenAI's prefix caching is *automatic*, with no marker; you just keep your prompts prefix-stable and read the `cached_tokens` field in usage.)

**OpenAI Batch API —** write your requests to a `.jsonl` (one per line, each with a `custom_id`), upload it, create the batch, poll, then download results. Half price, async, for work nobody's waiting on.

```python
# pip install openai  — reference only, not run here
from openai import OpenAI
client = OpenAI()

# 1. Build a .jsonl: one line per request, each tagged with a custom_id you'll
#    use to map results back (output order is NOT guaranteed to match input).
#    Line shape: {"custom_id": "...", "method": "POST",
#                 "url": "/v1/chat/completions",
#                 "body": {"model": "...", "messages": [...]}}

# 2. Upload it and kick off the batch.
f = client.files.create(file=open("requests.jsonl", "rb"), purpose="batch")
batch = client.batches.create(
    input_file_id=f.id,
    endpoint="/v1/chat/completions",
    completion_window="24h",          # the only supported window
)

# 3. Poll until done (minutes to 24h), then download the output file.
batch = client.batches.retrieve(batch.id)     # -> status: validating/in_progress/completed
if batch.status == "completed":
    results = client.files.content(batch.output_file_id).text  # .jsonl, one result per line
```

The economics from the napkin-math cell below are the whole reason to bother: ~50% off *everything*, stacking with a cheap model, for backfills and bulk evals. The cost is latency and a bit of plumbing (jsonl in, poll, jsonl out), trivial when no human is waiting.

## Napkin math you should be able to do cold

You've now met the three cost levers. Before the arithmetic, here they are side by side: what each attacks, its rough discount, and when it's the right reach:

| Lever | What it attacks | Rough effect | Tradeoff | Reach for it when |
|---|---|---|---|---|
| **Prompt caching** | the repeated *prefix* | ~0.1x on cached tokens (write ~1.25x once) | short TTL; exact-prefix-match discipline | a big system prompt / document repeats across calls |
| **Batch API** | *everything* in the request | ~0.5x on all tokens | async, minutes–24h latency | nobody's waiting: backfills, bulk evals, nightly jobs |
| **Model choice** | the *rate* itself | often 3–5x between tiers | smaller model may fail your eval | the cheap tier passes the same eval as the expensive one |

They **stack multiplicatively**: small model × batch × cached prefix is routinely 20–50x cheaper than a realtime large model with no cache. And every one of them reduces to the same formula: `tokens × rate`, with three multipliers (cache write ~1.25x input, cache read ~0.1x input, batch 0.5x everything). An AI engineer who can't sketch this on a napkin gets surprised by a bill; one who can spots the 10x saving in a design review.

The rates below are illustrative per-token prices (Groq's free tier is $0, and cache/batch aren't offered there); the point is the *shape* of the arithmetic, which is identical on every provider that bills per token. Helper below, then two worked examples.

In [7]:
PRICES = {
    # USD per million tokens: (input, output). Illustrative rates — the shape is what matters.
    'small':  (1.00, 5.00),    # cheap/bulk tier
    'mid':    (3.00, 15.00),   # iterate-on default
    'large':  (5.00, 25.00),   # reasoning-heavy tier
}


def estimate_cost(model, input_tokens=0, output_tokens=0,
                  cache_read=0, cache_write=0, batch=False):
    inp, out = PRICES[model]
    usd = (input_tokens * inp
           + output_tokens * out
           + cache_read * inp * 0.10
           + cache_write * inp * 1.25) / 1_000_000
    return usd * (0.5 if batch else 1.0)


# Example 1: a support bot. 10k requests/day, 8k-token runbook prefix, 200-token
# question, 300-token answer. Compare no-cache vs cached prefix (on the 'mid' tier).
n = 10_000
no_cache = n * estimate_cost('mid', input_tokens=8_200, output_tokens=300)
cached = (estimate_cost('mid', cache_write=8_000, input_tokens=200, output_tokens=300)
          + (n - 1) * estimate_cost('mid', cache_read=8_000, input_tokens=200, output_tokens=300))
print(f'support bot / day  no cache: ${no_cache:,.2f}   cached: ${cached:,.2f}   '
      f'saving: {1 - cached / no_cache:.0%}')

# Example 2: backfill-classify 1M tickets (500 in / 10 out each) on the small tier, batch vs realtime.
rt = 1_000_000 * estimate_cost('small', input_tokens=500, output_tokens=10)
bt = 1_000_000 * estimate_cost('small', input_tokens=500, output_tokens=10, batch=True)
print(f'1M-ticket backfill  realtime: ${rt:,.2f}   batch: ${bt:,.2f}')

# And the wrong-model version of example 2, because someone always proposes it:
large_rt = 1_000_000 * estimate_cost('large', input_tokens=500, output_tokens=10)
print(f'same backfill on realtime large tier: ${large_rt:,.2f}  <- this is the design-review catch')

support bot / day  no cache: $291.00   cached: $75.03   saving: 74%
1M-ticket backfill  realtime: $550.00   batch: $275.00
same backfill on realtime large tier: $2,750.00  <- this is the design-review catch


Run it and note the shape of the savings: caching attacks the *repeated prefix* (dominant when the shared context is big), batch attacks *everything* (dominant for bulk jobs), and model choice dwarfs both. The three multiply: a small model + batch + a cached prefix is routinely 20-50x cheaper than a realtime large model with no cache, for work where the cheap version passes the same eval.

## Exercises

1. Compare the `tiktoken` estimate to the true count: encode several prompts of different lengths with `count(...)`, send each with `max_tokens=1`, and compare the estimate to `usage.prompt_tokens`. How far off is tiktoken for the model you're calling?
2. Write `fit_history(history, budget_tokens)` that drops the oldest turns (always in user/assistant pairs) until `count(history)` fits the budget. Test it on a fabricated 20-turn conversation.
3. Extend `estimate_cost` to take a `calls_per_day` and `cache_hit_rate` and produce a monthly projection table for all three tiers, cached and uncached. Sanity-check one row by hand.
4. Measure Groq's real latency vs token count: time requests with `max_tokens` of 50, 200, 800 and confirm total latency scales with output tokens. This is the "reserve output headroom" lesson made concrete.